# Module 8: Rolling Statistics and Control Limits

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

"Is this month unusual?" is the most common question asked of public safety
data, and the hardest to answer casually, because unusual has to mean unusual
**compared to what**.

This module builds that comparison properly: a centre line that moves with the
trend and the season, and limits wide enough to allow for how much counts
genuinely vary.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

## 2. Rolling statistics

A rolling window summarises the recent past. The mean shows where the level is;
the standard deviation shows how much it has been moving.

In [ ]:
cedar = series("A002")

roll = pd.DataFrame({
    "observed": cedar,
    "rolling mean, 12 months": cedar.rolling(12).mean().round(1),
    "rolling spread, 12 months": cedar.rolling(12).std().round(1)})
roll.loc["2021-01":"2021-12"]

Look at what June 2021 does to both columns. The rolling mean jumps by about 12
incidents and stays elevated for a year, then drops back. The rolling spread
more than triples.

**A rolling window carries an event for as long as the window.** That is useful
for smoothing and useless for detection: by the time the mean has moved, the
event is eleven months old.

## 3. A first attempt: one fixed limit

The textbook control chart sets a centre line at the baseline mean and limits
three standard deviations away. For counts, the standard deviation is close to
the square root of the mean, so the limits are

    mean ± 3 × sqrt(mean)

In [ ]:
baseline = cedar[cedar.index.year <= 2020]
centre = baseline.mean()
upper = centre + 3 * np.sqrt(centre)

flagged = cedar[cedar > upper]
print(f"baseline mean {centre:.1f}, upper limit {upper:.1f}")
print("\nmonths above the limit:")
print(flagged.astype(int).to_string())

Three flags, and only one of them is an event. **July 2019 at 55 and June 2023
at 59 are ordinary summer months.** The limit does not know that summer exists,
so it will keep raising an alarm every warm month for as long as the chart is
used, and the people receiving those alarms will stop reading them.

The fixed limit also ignores the trend. Cedar Falls is declining, so a baseline
from 2019 and 2020 is too high for 2025 and the chart will never flag anything
at the low end even if the agency collapses.

## 4. A centre line that moves

Replace the flat baseline with the trend times the season from
[Module 5](Module_05_Decomposition.md). Now each month is compared against what
was expected **for that month, at that point in the agency's history**.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(np.log(cedar), period=12, robust=True).fit()
expected = np.exp(stl.trend + stl.seasonal)

upper = expected + 3 * np.sqrt(expected)
out = cedar[cedar > upper]

print("months above the moving limit:")
print(pd.DataFrame({"observed": cedar[out.index].astype(int),
                    "expected": expected[out.index].round(1),
                    "limit": upper[out.index].round(1)}).to_string())

July 2019 has dropped off the list, which is the improvement we were after.
Five months remain, and four of them are barely over.

## 5. Counts vary more than the square root rule allows

The `sqrt(mean)` rule assumes the variance of a count equals its mean. Real
administrative counts are **overdispersed**: incidents cluster, conditions
persist, and the variance runs higher.

Measure it, then widen the limits by the same factor.

In [ ]:
# leave out the documented unrest month when measuring ordinary spread
ordinary = ~((cedar.index.year == 2021) & (cedar.index.month == 6))

pearson = ((cedar - expected) ** 2 / expected)[ordinary]
phi = pearson.sum() / (ordinary.sum() - 1)

print(f"dispersion: {phi:.2f}")
print("1.0 would mean the square root rule is exactly right;")
print("above 1 means the counts move around more than it allows.")

In [ ]:
upper_wide = expected + 3 * np.sqrt(phi * expected)
lower_wide = (expected - 3 * np.sqrt(phi * expected)).clip(lower=0)

flags = cedar[(cedar > upper_wide) | (cedar < lower_wide)]
report = pd.DataFrame({
    "observed": cedar[flags.index].astype(int),
    "expected": expected[flags.index].round(1),
    "limit": upper_wide[flags.index].round(1)})
report["times the limit"] = (report["observed"] / report["limit"]).round(2)
report.sort_values("times the limit", ascending=False)

Three flags, and the ranking is the point. **June 2021 is more than three times
its limit. The other two clear it by less than a fifth.**

A control chart is a screening tool, not a verdict. It hands you a short list.
One item on that list is a documented week of civil unrest; the other two are
months worth a glance at the record and probably nothing more. Reporting them
as three equivalent alarms would be a failure of judgment, not of statistics.

## 6. Small agencies cannot support a monthly chart

Elkhorn averages under one incident a month. Its upper limit sits at about 3.5,
and in 88 months exactly one month reaches it, which is roughly what chance
alone produces.

In [ ]:
elk = series("A006")
mu = elk.mean()

print(f"mean {mu:.2f} a month, upper limit {mu + 3 * np.sqrt(mu):.2f}")
print(f"months at or above it: {int((elk >= mu + 3 * np.sqrt(mu)).sum())} of {len(elk)}")
print(f"months at zero: {int((elk == 0).sum())}")

There is no useful monthly chart here. The fix is to change the time unit until
the counts are large enough to say something, which is Beginner
[Topic 4](../Beginner/Topic_04_Time_Units_Frequency_And_Aggregation.md) applied
to a new purpose.

In [ ]:
yearly = elk.groupby(elk.index.year).sum().loc[2019:2025]
mu_y = yearly.mean()

print(yearly.astype(int).to_string())
print(f"\nyearly mean {mu_y:.1f}, upper limit {mu_y + 3 * np.sqrt(mu_y):.1f}")
print(f"years above it: {int((yearly > mu_y + 3 * np.sqrt(mu_y)).sum())}")

## 7. A reusable chart

In [ ]:
def control_chart(s, z=3, exclude=None):
    """Expected value from trend and season, with dispersion widened limits."""
    stl = STL(np.log(s.clip(lower=0.5)), period=12, robust=True).fit()
    expected = np.exp(stl.trend + stl.seasonal)

    mask = pd.Series(True, index=s.index)
    if exclude is not None:
        mask.loc[exclude] = False
    phi = (((s - expected) ** 2 / expected)[mask]).sum() / (mask.sum() - 1)

    spread = z * np.sqrt(phi * expected)
    out = pd.DataFrame({"observed": s, "expected": expected,
                        "lower": (expected - spread).clip(lower=0),
                        "upper": expected + spread})
    out["outside"] = (out["observed"] > out["upper"]) | (out["observed"] < out["lower"])
    out["times the limit"] = (out["observed"] / out["upper"]).round(2)
    out.attrs["dispersion"] = phi
    return out

In [ ]:
c = control_chart(series("A012"))
print(f"Grandview, dispersion {c.attrs['dispersion']:.2f}")
print(c[c["outside"]].round(1).to_string())

And here is why the `exclude` argument exists. Leave the unrest month in when
measuring Cedar Falls' ordinary spread and the dispersion is inflated more than
fivefold, the limits balloon, and **the chart stops being able to see anything
except the event that broke it**.

In [ ]:
with_event = control_chart(series("A002"))
without = control_chart(series("A002"), exclude=pd.Timestamp("2021-06-01"))

print(f"unrest month left in : dispersion {with_event.attrs['dispersion']:5.2f}, "
      f"{int(with_event['outside'].sum())} months flagged")
print(f"unrest month taken out: dispersion {without.attrs['dispersion']:5.2f}, "
      f"{int(without['outside'].sum())} months flagged")

## 8. What to carry away

| Habit | Why |
|---|---|
| Move the centre line with the trend and the season | a flat limit flags every summer forever |
| Measure the dispersion, do not assume it is 1 | administrative counts are overdispersed |
| Rank flags by how far past the limit they sit | in or out is not enough information |
| Treat a flag as a question | the chart screens; a person decides |
| Change the time unit for small agencies | a monthly chart of a tiny count detects nothing |
| Exclude known events when measuring ordinary spread | otherwise one event widens the limits and hides the next |

## Exercise

Run the chart on Harbor Point. The dataset records that this agency changed how
it classified calls in January 2023. Does a use of force chart notice?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A003"

if AGENCY:
    c = control_chart(series(AGENCY))
    print(f"dispersion {c.attrs['dispersion']:.2f}")
    flagged = c[c["outside"]]
    print(f"months flagged: {len(flagged)} of {len(c)}")
    print(flagged.round(1).to_string() if len(flagged) else "none")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A003"
```

Only two months are flagged, neither of them anywhere near January 2023, so
the reclassification leaves no mark at all.

That is the right answer and a useful warning. The change affected how **calls
for service** were labelled, moving them between the `Other` and
`Public Order Offense` categories. It did not touch use of force records, so a
use of force chart has nothing to detect.

A control chart only watches the series you point it at. Catching the
reclassification requires charting the affected category **and its total**,
which is Beginner [Topic 19](../Beginner/Topic_19_Data_Quality_And_Pitfalls.md).
Monitoring is a choice about what to monitor before it is a choice about method.

</details>

---

**Next:** Part III of the Intermediate series, on comparing and relating series:
year over year and indexing, reading autocorrelation, lead and lag between two
series, and comparing agencies fairly.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*